# Topic 16: Data Preparation for Machine Learning

Before training any ML model, data must be properly prepared.

**Topics:** Train-test split · Feature selection · Scaling · Encoding

```bash
pip install scikit-learn
```


In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
np.random.seed(42)

n = 300
df = pd.DataFrame({
    "Age":        np.random.randint(22,60,n),
    "Experience": np.random.randint(0,30,n),
    "Salary":     np.random.normal(60000,15000,n).clip(25000,120000).round(-2),
    "Score":      np.random.normal(70,12,n).clip(30,100).round(1),
    "Gender":     np.random.choice(["Male","Female"],n),
    "Education":  np.random.choice(["BSc","MSc","PhD"],n),
    "Promoted":   np.random.choice([0,1],n,p=[0.65,0.35]),
})
df.loc[(df["Salary"]>75000)&(df["Experience"]>10),"Promoted"]=1
print(df.head()); print(df.shape)

## 1. Train-Test Split

> Split data so the model is **trained** on one part and **tested** on a completely unseen part.


In [ ]:
X = df[["Age","Experience","Salary","Score"]]
y = df["Promoted"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training : {len(X_train)} rows")
print(f"Test     : {len(X_test)} rows")
print(f"\nClass distribution in train:")
print(y_train.value_counts(normalize=True).round(3))

## 2. Feature Scaling

Scaling ensures no single feature dominates just because its values are larger.

| Method | Formula | Output |
|---|---|---|
| Min-Max | (x - min) / (max - min) | [0, 1] |
| Z-score | (x - mean) / std | mean=0, std=1 |


In [ ]:
# IMPORTANT: fit on training data ONLY, then transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # use same scaler!

print("Before scaling — Salary range:", X_train["Salary"].min(), "to", X_train["Salary"].max())
print("After scaling  — Salary range:", X_train_scaled[:,2].min().round(2),
      "to", X_train_scaled[:,2].max().round(2))

## 3. Encoding Categorical Variables

ML models need numbers — text categories must be converted.

| Method | When to use |
|---|---|
| Label Encoding | Ordinal (BSc < MSc < PhD) |
| One-Hot Encoding | Nominal (no order: city, gender) |


In [ ]:
# One-Hot Encoding (get_dummies)
gender_ohe = pd.get_dummies(df["Gender"], prefix="Gender", drop_first=True)
print("One-Hot Encoded Gender:\n", gender_ohe.head())

# Label Encoding for Education (has natural order)
edu_map = {"BSc": 0, "MSc": 1, "PhD": 2}
df["Education_Enc"] = df["Education"].map(edu_map)
print("\nEducation encoded:\n", df[["Education","Education_Enc"]].drop_duplicates())

## 4. Full Preparation Pipeline


In [ ]:
from sklearn.preprocessing import StandardScaler

# Encode
df["Gender_Enc"] = (df["Gender"] == "Male").astype(int)
dept_dummies = pd.get_dummies(df.get("Department", pd.Series(["A"]*n)), prefix="Dept", drop_first=True)

features = ["Age","Experience","Salary","Score","Gender_Enc","Education_Enc"]
X_all = df[features]
y_all = df["Promoted"]

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr[["Age","Experience","Salary","Score"]] = sc.fit_transform(X_tr[["Age","Experience","Salary","Score"]])
X_te[["Age","Experience","Salary","Score"]] = sc.transform(X_te[["Age","Experience","Salary","Score"]])

print(f"Ready for ML!  Train: {X_tr.shape}  Test: {X_te.shape}")

## ✅ Summary

1. **Split first** — then fit the scaler on training data only
2. **Scale numeric** features before algorithms that care about distance
3. **Label-encode** ordinal categories (with natural order)
4. **One-hot-encode** nominal categories (no natural order)
5. Never let test data influence training preprocessing
